In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from datetime import datetime

def parse(x):
    return datetime.strptime('190'+x, '%Y-%m')

# Load the data
df = pd.read_csv('shampoo_sales.csv', header=0, parse_dates=[0], index_col=0, date_parser=parse).squeeze()

# Convert Series to DataFrame
df = df.to_frame(name='Sales')

# Normalize the data
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(df)

# Create sequences for time series prediction
def create_sequences(data, seq_length):
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:(i + seq_length), 0])
        y.append(data[i + seq_length, 0])
    return np.array(X), np.array(y)

# Set the sequence length (lookback period)
seq_length = 3  # Number of time steps to look back

# Generate input sequences and target values
X, y = create_sequences(scaled_data, seq_length)

# Split the data
train_size = int(len(X) * 0.7)
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

# Reshape input to be [samples, time steps, features]
X_train = np.reshape(X_train, (X_train.shape[0], X_train.shape[1], 1))
X_test = np.reshape(X_test, (X_test.shape[0], X_test.shape[1], 1))

# Convert data to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

# Create Dataset and DataLoader
class TimeSeriesDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = TimeSeriesDataset(X_train_tensor, y_train_tensor)
test_dataset = TimeSeriesDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)


In [ ]:

class RNNModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(RNNModel, self).__init__()
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)  # RNN layer
        self.fc = nn.Linear(hidden_size, output_size)  # Fully connected layer

    def forward(self, x):
        out, _ = self.rnn(x)  # RNN outputs and hidden state
        out = self.fc(out[:, -1, :])  # Use the last time step's output
        return out


In [ ]:

# Define the GRU model
class GRUModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(GRUModel, self).__init__()
        self.gru = nn.GRU(input_size, hidden_size, batch_first=True)  # GRU layer
        self.fc = nn.Linear(hidden_size, output_size)  # Fully connected layer

    def forward(self, x):
        out, _ = self.gru(x)  # GRU outputs and hidden state
        out = self.fc(out[:, -1, :])  # Use the last time step's output
        return out


In [ ]:

# Define the LSTM model
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)  # LSTM layer
        self.fc = nn.Linear(hidden_size, output_size)  # Fully connected layer

    def forward(self, x):
        out, (h_n, c_n) = self.lstm(x)  # LSTM outputs, hidden state, and cell state
        out = self.fc(out[:, -1, :])  # Use the last time step's output
        return out


In [ ]:
    
# Model parameters
input_size = 1  # One feature
hidden_size = 50  # 50 RNN units
output_size = 1  # Predicting one value
model = RNNModel(input_size, hidden_size, output_size)

# Loss function and optimizer
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# Training loop
num_epochs = 100
for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        y_pred = model(X_batch)
        loss = criterion(y_pred, y_batch)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    print(f"Epoch {epoch + 1}/{num_epochs}, Train Loss: {train_loss / len(train_loader):.4f}")


In [ ]:

# Evaluation
model.eval()
test_loss = 0.0
y_train_pred = []
y_test_pred = []

with torch.no_grad():
    for X_batch, y_batch in train_loader:
        y_pred = model(X_batch)
        y_train_pred.append(y_pred.numpy())
    for X_batch, y_batch in test_loader:
        y_pred = model(X_batch)
        y_test_pred.append(y_pred.numpy())

y_train_pred = np.concatenate(y_train_pred)
y_test_pred = np.concatenate(y_test_pred)

# Inverse transform predictions
y_train_inv = scaler.inverse_transform(y_train.reshape(-1, 1))
train_predict = scaler.inverse_transform(y_train_pred)
y_test_inv = scaler.inverse_transform(y_test.reshape(-1, 1))
test_predict = scaler.inverse_transform(y_test_pred)

# Calculate RMSE
train_rmse = np.sqrt(np.mean((train_predict - y_train_inv) ** 2))
test_rmse = np.sqrt(np.mean((test_predict - y_test_inv) ** 2))

print(f"Train RMSE: {train_rmse:.3f}")
print(f"Test RMSE: {test_rmse:.3f}")
